# Entraînement PPO : rebonds successifs

Choisir le kernel **so101-robotic-learning**, puis exécuter les cellules dans l'ordre.
EGL n'est pas imposé : sur un serveur Linux sans écran, configurer le backend avant tout import MuJoCo si nécessaire.

La récompense vaut **5 × rebonds + 0.05 × shaping − 5 en cas d'échec − pénalité d'énergie − pénalité de distance au segment** (chute, raquette au sol ou balle à plus de 2 m de la raquette). La réussite demande
5 rebonds consécutifs par défaut. Les checkpoints sont sélectionnés selon les métriques
physiques ; leurs scores ne sont pas directement comparables aux anciens entraînements.
Les logs sont écrits dans `runs/ppo_bounce/`. Modèles, rapports et vidéos utilisent des sous-dossiers datés distincts.


In [ ]:
from datetime import datetime
from pathlib import Path
import json
import numpy as np
from IPython.display import JSON, Markdown, display

# Works from the repository root or any of its subdirectories.
root = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "pyproject.toml").is_file()
             and (path / "src/so101_robotic_learning").is_dir()), None)
if root is None:
    raise RuntimeError("Ouvrir ce notebook depuis le dépôt ou l'un de ses sous-dossiers.")
xml_path = root / "assets/mjcf/so101_new_calib copy.xml"
if not xml_path.is_file():
    raise FileNotFoundError(xml_path)

# MuJoCo selects the platform backend. Do not force EGL on Windows.
from so101_robotic_learning.bounce_rl.env.bounce_events import BounceConfig
from so101_robotic_learning.bounce_rl.env.gravity_curriculum import GravityCurriculumConfig
from so101_robotic_learning.bounce_rl.rewards import (
    RewardMultiObjectif, BounceReward, FailurePenalty, EnergyPenalty, BallSegmentPenalty,
)

from so101_robotic_learning.bounce_rl.rl import train_ppo

from so101_robotic_learning.bounce_rl.rewards.reward_ball_aligned_on_z_and_above_paddle import BallAlignedOnZAndAbovePaddleReward


## Paramètres

`reward` définit l’objectif complet. Ses poids sont configurés dans le composite. Les paramètres d’environnement
passent par `env_kwargs`. Pour utiliser un historique, modifier `latent_steps` **avant** de
lancer l'entraînement et conserver exactement la même configuration lors du test du modèle.
PPO et SAC utilisent ici la même tâche et le même seuil de réussite.

Curriculum facultatif : remplacer `train_config["gravity_curriculum"] = None` par
`GravityCurriculumConfig(start_gravity=3.0, gravity_step=1.0, target_gravity=9.81, success_bounces=5)`.
Après 5 rebonds consécutifs ou un épisode arrivé à sa durée maximale sans échec,
le palier suivant est appliqué au prochain reset. `success_on_timeout=True` active ce second critère
par défaut. En évaluation, il valide aussi la réussite sans faire avancer le palier.
Une évaluation arrêtée avant la limite de l’environnement ne valide pas la survie.
Un épisode garde sa gravité. Les workers partagent la progression, les évaluations ne la modifient pas.
La gravité et les paliers sont enregistrés dans TensorBoard. Les checkpoints gardent un fichier
`*_curriculum.json` à conserver avec le `.zip` pour reprendre au bon palier.
La progression s’arrête à 9.81 m/s². La reward et les conditions de rebond restent inchangées.


In [ ]:
run_id = "ppo_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
run_root = root

env_kwargs = {
    "camera_name": "gripper_cam",  # policy input only
    "render_profile": "train_fast",  # same profile for training and evaluation
    "video_render_profile": "standard",  # external camera only
    "video_camera_name": "overview_cam",  # visualization only
    "video_width": 320,
    "video_height": 240,
    "bounce_config": BounceConfig(success_bounces=5, max_ball_paddle_distance=2.0),
    "ball_segment_start_height": 0.2,  # lower endpoint above paddle_face (m)
    "ball_segment_length": 1.0,  # segment length along world Z (m)
    "max_episode_steps": 1024,
    "frame_skip": 20,
    "latent_steps": np.array([], dtype=np.int32),
}
reward = (
    RewardMultiObjectif()
    .add_reward("bounces", BounceReward(), weight=5.0)
    .add_reward("shaping", BallAlignedOnZAndAbovePaddleReward(), weight=0.05)
    .add_penalty("failure", FailurePenalty(), weight=5.0)
    .add_penalty("energy", EnergyPenalty(), weight=0.01)
    .add_penalty("ball_segment", BallSegmentPenalty(), weight=1.0)
)

train_config = {
    "xml_path": xml_path,
    "root": run_root,
    "model_dir_name": f"models/{run_id}",
    "env_kwargs": env_kwargs,
    "reward": reward,
    "total_timesteps": 20_000,  # additional steps when resuming
    "gravity_curriculum": None,  # GravityCurriculumConfig() to enable (3 -> 9.81 m/s²)
    "resume_from": None,  # e.g. root / "models/previous_run/ppo_bounce_last.zip",
    "nb_vec_env_train": 1,
    "nb_vec_env_test": 1,
    "learning_rate": 3e-4,
    "n_steps": 256,
    "batch_size": 64,
    "n_epochs": 10,
    "eval_every_episodes": 10,
    "n_eval_episodes": 3,
    "eval_max_steps": env_kwargs["max_episode_steps"],
    "train_render_mode": None,
    "eval_render_mode": None,  # recording is enabled independently of the viewer
    "save_eval_video": True,
    "tensorboard_log": True,
    "run_subdir": "ppo_bounce",
    "device": "auto",
}
RUN_TENSORBOARD = True  # optional inline dashboard
RUN_POLICY_TEST = False  # optional evaluation of the best checkpoint
print(f"Logs TensorBoard : {run_root / 'runs/ppo_bounce'}")
print(f"Modèles et rapports : {run_root / train_config['model_dir_name']}")


## Entraînement

Pour reprendre, renseigner `train_config["resume_from"]` avec un checkpoint `.zip`.
`total_timesteps` désigne alors les pas supplémentaires. L’architecture et les hyperparamètres
proviennent du checkpoint, les valeurs correspondantes du notebook sont ignorées.
Conserver des observations et actions compatibles et un nouveau `model_dir_name` pour cette session.
Un nouveau run TensorBoard utilise les pas cumulés. Les compteurs d’épisodes et le classement
du meilleur modèle repartent pour cette session. Un rollout PPO partiel n’est pas restauré.


In [ ]:
train_ppo.train(**train_config)
